In [14]:
import os

from dotenv import load_dotenv

# ──────────────────────────────────────────────
# 환경변수
# ──────────────────────────────────────────────
load_dotenv(override = True)

# OPENROUTER SET
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL      = "https://openrouter.ai/api/v1"
# MODEL_NAME          = "openai/gpt-oss-120b:free"
# MODEL_NAME          = "nvidia/nemotron-3-super-120b-a12b:free"

# ELICE SET
ELICE_API_KEY       = os.getenv("ELICE_API_KEY")
ELICE_URL           = os.getenv("ELICE_GLM_URL")
MODEL_NAME          = "zai-org/glm-4.7"

# EMBEDDING ~ ELICE
ELICE_EMBEDDING_URL   = os.getenv("ELICE_EMBEDDING_URL")
ELICE_EMBEDDING_MODEL = os.getenv("ELICE_EMBEDDING_MODEL")

# OTHER TOOLS
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model    = MODEL_NAME,
    api_key  = ELICE_API_KEY,
    base_url = ELICE_URL,
)

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=ELICE_EMBEDDING_MODEL,
    openai_api_key=ELICE_API_KEY,
    openai_api_base=ELICE_EMBEDDING_URL
)

# LangChain을 활용한 챗봇 구현

### 1. Retrieval augmented generation

- 문서를 바탕으로 질의 응답을 할 수 있는 봇을 만듭니다.

### 2. 대화 히스토리를 고려한 RAG 챗봇

- 단발성 질문이 아닌, 과거 질문 내역을 바탕으로 질문을 이어갈 수 있게끔 history를 추가합니다.

## 1. Retrieval augmented generation

다음과 같은 기능을 가진 챗봇을 만들 예정입니다.

1. 문서를 기반으로 관련성 있는 정보를 가져오는 봇
2. 관련 정보를 바탕으로 사용자의 질문에 답변을 하는 봇

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Maximizing Muscle Hypertrophy.pdf")
pages = loader.load_and_split()

- PDF를 읽어 LangChain의 다른 컴포넌트에서 사용할 수 있는 Document의 형태로 변환합니다.
- 읽는 파일이나 데이터의 종류에 따라 사용해야 하는 loader가 달라집니다.

In [3]:
pages[0]

Document(metadata={'producer': 'iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2019-12-06T03:19:50+01:00', 'moddate': '2019-12-06T03:19:50+01:00', 'source': 'Maximizing Muscle Hypertrophy.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='International  Journal  of \nEnvironmental Research\nand Public Health\nReview\nMaximizing Muscle Hypertrophy: A Systematic\nReview of Advanced Resistance Training Techniques\nand Methods\nMichal Krzysztoﬁk *\n , Michal Wilk\n , Grzegorz Wojdała\n and Artur Goła´ s\nInstitute of Sport Sciences, Jerzy Kukuczka Academy of Physical Education in Katowice, ul. Mikolowska 72a,\n40-065 Katowice, Poland; m.wilk@awf.katowice.pl (M.W.); wojdala.grzegorz@gmail.com (G.W.);\na.golas@awf.katowice.pl (A.G.)\n* Correspondence: m.krzysztoﬁk@awf.katowice.pl\nReceived: 12 October 2019; Accepted: 3 December 2019; Published: 4 December 2019\n/gid00030/gid00035/gid00032/gid00030/gid00038/gid00001/gid00033/gid

- 텍스트를 LLM에 활용하기 위해서는 사용자의 '질문'과 연관된 정보를 외부에서 제공해줄 수 있어야 합니다.
- '질문'과의 연관성을 측정하기 위한 방법이 embedding 유사도 측정입니다.
- 텍스트를 벡터의 형태로 변환하고, 벡터간의 similarity를 측정함으로써 두 텍스트의 '연관성'을 숫자로 표현할 수 있습니다.
- 따라서 활용하고자 하는 파일에 있는 텍스트를 벡터로 변환하고 저장해야 활용할 수 있습니다.
    - 가장 첫 번째 단계는 텍스트를 split하여 저장하는 것입니다.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
splits = text_splitter.split_documents(pages)

In [6]:
splits[0]

Document(metadata={'producer': 'iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2019-12-06T03:19:50+01:00', 'moddate': '2019-12-06T03:19:50+01:00', 'source': 'Maximizing Muscle Hypertrophy.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='International  Journal  of \nEnvironmental Research\nand Public Health\nReview\nMaximizing Muscle Hypertrophy: A Systematic\nReview of Advanced Resistance Training Techniques\nand Methods\nMichal Krzysztoﬁk *\n , Michal Wilk\n , Grzegorz Wojdała\n and Artur Goła´ s\nInstitute of Sport Sciences, Jerzy Kukuczka Academy of Physical Education in Katowice, ul. Mikolowska 72a,\n40-065 Katowice, Poland; m.wilk@awf.katowice.pl (M.W.); wojdala.grzegorz@gmail.com (G.W.);\na.golas@awf.katowice.pl (A.G.)\n* Correspondence: m.krzysztoﬁk@awf.katowice.pl\nReceived: 12 October 2019; Accepted: 3 December 2019; Published: 4 December 2019\n/gid00030/gid00035/gid00032/gid00030/gid00038/gid00001/gid00033/gid

<span style="background-color: #A5B68D; border-radius: 5px; padding: 2px 6px; border: 1px solid #ccc; font-family: sans-serif;">
    retriever</span>에는 아주 많은 기능이 내포되어 있습니다.
코드를 실행함으로서 다음 스텝들이 내부에서 실행됩니다.

1. Document 내에 있는 text를 추출
2. 추가할 meta data가 있다면 추가
3. Text를 embedding vector로 변환 (이때 embedding model 활용)
4. Embedding vector를 vectorDB에 적재

심화된 내용을 공부하고 싶으시면 : Retrieval augmented generation (RAG)

In [ ]:
# 패키지는 uv로 관리됩니다: uv add langchain-chroma chromadb

In [ ]:
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [15]:
vectorstore = Chroma.from_documents(
    documents = splits,
    embedding = embeddings,
)

retriever   = vectorstore.as_retriever()

LCEL(LangChain Expression Language)에서는 `|` 연산자로 컴포넌트를 직접 연결합니다.

정보 흐름: `retriever` → `format_docs` → `prompt` → `llm` → `StrOutputParser`

- `{"context": retriever | format_docs, "input": RunnablePassthrough()}` : retriever가 문서를 검색하고 텍스트로 합친 뒤 `context`에 담고, 원래 질문은 `input`으로 그대로 전달합니다.
- `format_docs` : Document 리스트를 하나의 문자열로 변환합니다.
- `StrOutputParser()` : LLM 출력(AIMessage)에서 문자열만 추출합니다.

In [ ]:
system_prompt = (
    """당신은 질문-답변을 담당하는 전문가 입니다. 다음 정보를 활용하여 질문에 답을 하시오. 
    모르면 모른다고 답하고, 답변은 간결하게 하시오.
    {context}"""
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("현재 논문의 주제가 뭐야?")

LCEL 파이프라인의 흐름:

```
질문(str)
  → {"context": retriever | format_docs, "input": RunnablePassthrough()}
  → prompt   # system + human 메시지로 완성
  → llm      # AIMessage 반환
  → StrOutputParser()  # 문자열 추출
```

`invoke()`에 문자열을 바로 넣으면 되고, 반환값도 바로 문자열입니다. (`dict["answer"]` 불필요)

## 2. 대화 히스토리를 고려한 RAG 챗봇

In [18]:
from langchain_core.runnables.history import RunnableWithMessageHistory

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "논문 리뷰 전문가 입니다. 사용자의 질문에 답하세요. {context}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# RunnablePassthrough.assign: 기존 키(input, chat_history)를 유지하면서 context를 추가
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["input"]))
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [ ]:
conversational_rag_chain.invoke(
    {"input": "논문의 주제가 뭐야?"},
    config={
        "configurable": {"session_id": "paperbot"}
    },
)

In [ ]:
conversational_rag_chain.invoke(
    {"input": "논문에서 이야기 하는 근육을 키우는 가장 좋은 방법은 뭐야?"},
    config={
        "configurable": {"session_id": "paperbot"}
    },
)

In [24]:
store

{'paperbot': InMemoryChatMessageHistory(messages=[HumanMessage(content='논문의 주제가 뭐야?', additional_kwargs={}, response_metadata={}), AIMessage(content='제시된 텍스트에 따르면, 이 논문은 **발저항 훈련 기법 및 방법(advanced resistance training techniques and methods)이 근비대(muscle hypertrophy)와 훈련 변수에 미치는 영향**을 조사한 체계적 문헌 고찰(Systematic Review)입니다.\n\n구체적으로 다음과 같은 내용을 다루고 있습니다:\n\n*   **주제:** 고급 저항 훈련 기법이 근육 비대와 훈련 관련 변인(변수)들에 미치는 효과 분석.\n*   **참여 대상:** 19세에서 44세 사이의 건강한 성인.\n*   **평가 지표:**\n    *   근비대 측정: MRI, DXA, 초음파 영상 등을 통한 근육 변화.\n    *   근력 측정: 1RM(최대 반복 횟수) 또는 5RM 등을 활용한 반복 최대 테스트.\n    *   훈련량: 반복 횟수, 총 부하, 근력 실패까지의 시간 등.\n*   **연구 범위:** 총 30건의 연구가 선정 기준에 부합하여 포함되었습니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='논문에서 이야기 하는 근육을 키우는 가장 좋은 방법은 뭐야?', additional_kwargs={}, response_metadata={}), AIMessage(content='제공해주신 텍스트 내에서 저자가 밝히고 있는 **근육을 키우는(근비대) 가장 핵심적인 방법**은 다음과 같습니다.\n\n**1. 핵심 요소: 높은 훈련 볼륨 (High Training Volume)**\n텍스트에 따르면, 저자는 훈련 볼륨(sets $\\times$ repetitions)과 사용하는 하중(loa